In [0]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [0]:
spark = SparkSession.builder.getOrCreate()
order_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true") 
    .load("/Volumes/orders/default/orders_csv/orders_100mb.csv")
)
order_df.show(5)
order_df.printSchema()

In [0]:
product_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true") 
    .load("/Volumes/orders/default/orders_csv/products.csv")
)
product_df.show(5)


Q1) Using the orders and products datasets, find the **top 2 most profitable products in each state**, where profit is defined as `(unit_price − cost_price) * quantity`. Consider only delivered orders. Return state, product_id, total profit, and rank within the state.


In [0]:
order_df_filtered = (
    order_df
    .filter(F.col("order_status") == 'DELIVERED')
    .select("product_id", "state", "price", "quantity")
)
product_df_filtered = product_df.select(F.col("product_id"), F.col("cost_price"))

df_merged = (
    order_df_filtered.join(F.broadcast(product_df_filtered), "product_id")
    .withColumn("profit",(F.col("price") - F.col("cost_price"))*F.col("quantity"))
    .groupby(F.col("state"),F.col("product_id"))
    .agg(
        F.sum(F.col("profit")).alias("total_profit")
    )
)
w = Window.partitionBy(F.col("state")).orderBy(F.col("total_profit").desc())

df_merged = (
    df_merged.withColumn("rank_within_state",F.row_number().over(w))
    .filter(F.col("rank_within_state") < 3)
)
df_merged.show()

Q2) For each customer, identify their **most recent order** and calculate the **difference between that order’s unit price and the average unit price of all products purchased by that customer**. Return customer_id, order_id, order_date, unit_price, and the price difference.

In [0]:
w = Window.partitionBy("customer_id").orderBy(F.col("order_date").desc(), F.col("order_id").desc())

order_df_filtered = (
          order_df.select("customer_id","order_id","order_date",'price')
          .withColumn("customer_ranked", F.row_number().over(w))
          .filter(F.col("customer_ranked") == 1)
)
avg_price_df = (
          order_df.groupby('customer_id')
          .agg(
            F.avg("price").alias("avg_unit_price")
          )
)
df_merged  = (
        order_df_filtered.join(avg_price_df,"customer_id","inner")
        .withColumn("price_difference",F.col("price") - F.col("avg_unit_price"))
        .select("customer_id","order_id","order_date","price","price_difference")
)
df_merged.show()

#m2

In [0]:
w_latest = Window.partitionBy("customer_id") \
                 .orderBy(F.col("order_date").desc(), F.col("order_id").desc())

w_avg = Window.partitionBy("customer_id")

df = (
    order_df
    .withColumn("avg_unit_price", F.avg("price").over(w_avg))
    .withColumn("rank", F.row_number().over(w_latest))
    .filter(F.col("rank") == 1)
    .withColumn("price_difference", F.col("price") - F.col("avg_unit_price"))
    .select("customer_id", "order_id", "order_date", "price", "price_difference")
)

df.show()

Q3)Using orders and products data, determine for each year the **product that contributed the highest percentage of total yearly revenue**. Return order_year, product_id, product_revenue, total_year_revenue, and revenue_percentage.

In [0]:
w = Window.partitionBy("order_year")
w_rank = Window.partitionBy("order_year").orderBy(F.col("revenue_percentage").desc())
df = (
    order_df
    .withColumn("product_revenue", F.col("quantity")* F.col("price"))
    .withColumn("order_year",F.year(F.col("order_date")))
    .select("order_year",'product_id','product_revenue')
    .groupby(["order_year",'product_id'])
    .agg(
      F.sum(F.col("product_revenue")).alias("product_revenue")
    )
    .withColumn('total_year_revenue',F.sum("product_revenue").over(w))
    .withColumn("revenue_percentage", F.col("product_revenue")*100.0/F.col("total_year_revenue"))
    .withColumn("ranked",F.row_number().over(w_rank))
    .filter(F.col("ranked")== 1)
    .select("order_year",'product_id',"product_revenue","total_year_revenue","revenue_percentage")
)
df.show()